# Customer Churn Analysis - Model Training and Evaluation

This notebook trains multiple machine learning models, performs hyperparameter tuning, and evaluates model performance.

## Contents
1. Data Loading and Preparation
2. Baseline Models Training
3. Advanced Models (XGBoost, LightGBM)
4. Hyperparameter Tuning
5. Model Comparison
6. SHAP Explainability
7. Business Impact Analysis
8. Final Model Selection

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from src.config import PROCESSED_DATA_PATH, MODEL_DIR, RANDOM_STATE, TEST_SIZE
from src.data_preprocessing import preprocess_pipeline, scale_features, prepare_data
from src.model_training import ChurnModelTrainer, train_and_evaluate_pipeline
from src.visualization import (
    plot_confusion_matrix, plot_roc_curve,
    plot_feature_importance, plot_model_comparison
)
from src.utils import explain_model_shap, print_model_summary, generate_model_report

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and Preparation

In [ ]:
# Load and preprocess data
print("Loading and preprocessing data...")
X, y = preprocess_pipeline(save_processed=True)

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nClass distribution:")
print(y.value_counts(normalize=True))

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nChurn rate in training: {y_train.mean()*100:.2f}%")
print(f"Churn rate in test: {y_test.mean()*100:.2f}%")

In [ ]:
# Scale features
X_train_scaled, X_test_scaled = scale_features(X_train, X_test, save_scaler=True)
print(f"\nFeatures scaled and scaler saved.")
print(f"Scaled training set shape: {X_train_scaled.shape}")
print(f"Scaled test set shape: {X_test_scaled.shape}")

## 2. Baseline Models Training

Train multiple baseline models to establish performance benchmarks:

In [ ]:
# Initialize trainer
trainer = ChurnModelTrainer(random_state=RANDOM_STATE)
trainer.initialize_models()

print("Models initialized:")
for name in trainer.models.keys():
    print(f"  - {name}")

In [ ]:
# Train all models
print("\nTraining all models...\n")
results_df = trainer.train_all_models(X_train_scaled, y_train, X_test_scaled, y_test)

print("\n" + "="*80)
print_model_summary(results_df)

In [ ]:
# Display results table
results_df.round(4)

## 3. Model Performance Visualization

In [ ]:
# Compare models by different metrics
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    results_sorted = results_df.sort_values(metric, ascending=False)
    ax = axes[idx]
    
    bars = ax.barh(results_sorted['model'], results_sorted[metric], color='steelblue')
    
    # Color the best model
    bars[0].set_color('coral')
    
    ax.set_xlabel(metric.replace('_', ' ').title(), fontsize=12)
    ax.set_title(f'Model Comparison - {metric.replace("_", " ").title()}', 
                 fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    # Add value labels
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, 
                f'{width:.3f}', ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ROC-AUC comparison
if 'roc_auc' in results_df.columns:
    plot_model_comparison(results_df, metric='roc_auc', save=True)

## 4. Best Model Evaluation

In [ ]:
# Get best model
best_model_name = trainer.best_model_name
best_model = trainer.best_model

print(f"Best Model: {best_model_name}")
print(f"\nModel Parameters:")
print(best_model.get_params())

In [ ]:
# Predictions
y_pred = best_model.predict(X_test_scaled)
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# Confusion Matrix
plot_confusion_matrix(
    y_test, y_pred,
    labels=['No Churn', 'Churn'],
    title=f'Confusion Matrix - {best_model_name}',
    save=True,
    save_name='confusion_matrix_best_model.png'
)

In [ ]:
# ROC Curve
plot_roc_curve(
    y_test, y_pred_proba,
    model_name=best_model_name,
    save=True,
    save_name='roc_curve_best_model.png'
)

## 5. Feature Importance Analysis

In [ ]:
# Feature importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    feature_names = X.columns.tolist()
    importances = best_model.feature_importances_
    
    plot_feature_importance(
        feature_names, importances,
        top_n=20,
        title=f'Feature Importance - {best_model_name}',
        save=True,
        save_name='feature_importance_best_model.png'
    )
    
    # Display top features
    feat_imp_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print("\nTop 20 Most Important Features:")
    print(feat_imp_df.head(20).to_string(index=False))
else:
    print(f"{best_model_name} does not have feature_importances_ attribute.")

## 6. SHAP Explainability Analysis

In [ ]:
# SHAP Analysis
print("Generating SHAP explanations...")
print("This may take a few minutes...\n")

feature_names = X.columns.tolist()

# Summary plot
explainer = explain_model_shap(
    best_model,
    X_train_scaled[:100],  # Use subset for faster computation
    X_test_scaled[:100],
    feature_names,
    plot_type='summary',
    max_display=20,
    save=True
)

In [ ]:
# SHAP bar plot
explainer = explain_model_shap(
    best_model,
    X_train_scaled[:100],
    X_test_scaled[:100],
    feature_names,
    plot_type='bar',
    max_display=20,
    save=True
)

## 7. Hyperparameter Tuning (Optional)

Fine-tune XGBoost and LightGBM for optimal performance:

In [ ]:
# Uncomment to perform hyperparameter tuning (takes longer)

# print("Starting hyperparameter tuning for XGBoost...")
# best_xgb, xgb_params = trainer.hyperparameter_tuning_xgboost(
#     X_train_scaled, y_train, method='random'
# )

# print("\nStarting hyperparameter tuning for LightGBM...")
# best_lgb, lgb_params = trainer.hyperparameter_tuning_lightgbm(
#     X_train_scaled, y_train, method='random'
# )

# # Evaluate tuned models
# tuned_xgb_metrics = trainer.evaluate_model(best_xgb, X_test_scaled, y_test, 'XGBoost (Tuned)')
# tuned_lgb_metrics = trainer.evaluate_model(best_lgb, X_test_scaled, y_test, 'LightGBM (Tuned)')

# print("\nTuned Models Performance:")
# print(f"XGBoost (Tuned): {tuned_xgb_metrics}")
# print(f"LightGBM (Tuned): {tuned_lgb_metrics}")

print("Hyperparameter tuning skipped. Uncomment the code above to run.")

## 8. Business Impact Analysis

In [ ]:
# Calculate business impact
business_metrics = trainer.calculate_business_impact(
    y_test, y_pred,
    avg_customer_value=1000,  # Adjust based on actual business data
    retention_cost=100
)

print("\n" + "="*80)
print("BUSINESS IMPACT ANALYSIS")
print("="*80)

print(f"\n📊 Confusion Matrix Breakdown:")
print(f"  True Positives (Correctly identified churners): {business_metrics['true_positives']}")
print(f"  False Positives (Unnecessary retention efforts): {business_metrics['false_positives']}")
print(f"  False Negatives (Missed churners): {business_metrics['false_negatives']}")
print(f"  True Negatives (Correctly identified non-churners): {business_metrics['true_negatives']}")

print(f"\n💰 Financial Impact:")
print(f"  Cost of Missed Churners: ${business_metrics['cost_of_missed_churners']:,.2f}")
print(f"  Cost of Retention Efforts: ${business_metrics['cost_of_retention']:,.2f}")
print(f"  Value of Retained Customers: ${business_metrics['value_retained']:,.2f}")
print(f"  Net Benefit: ${business_metrics['net_benefit']:,.2f}")
print(f"  ROI: {business_metrics['roi']:.2f}%")

In [ ]:
# Visualize business impact
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cost breakdown
costs = [
    business_metrics['cost_of_missed_churners'],
    business_metrics['cost_of_retention']
]
labels = ['Missed Churners', 'Retention Efforts']
colors = ['#ff6b6b', '#4ecdc4']

axes[0].bar(labels, costs, color=colors)
axes[0].set_title('Cost Breakdown', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cost ($)', fontsize=12)
for i, v in enumerate(costs):
    axes[0].text(i, v, f'${v:,.0f}', ha='center', va='bottom', fontsize=10)

# Net benefit
categories = ['Costs', 'Revenue Saved', 'Net Benefit']
values = [
    -(business_metrics['cost_of_missed_churners'] + business_metrics['cost_of_retention']),
    business_metrics['value_retained'],
    business_metrics['net_benefit']
]
colors_benefit = ['#ff6b6b', '#95e1d3', '#38ada9']

bars = axes[1].bar(categories, values, color=colors_benefit)
axes[1].set_title('Financial Summary', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Amount ($)', fontsize=12)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

for i, (bar, v) in enumerate(zip(bars, values)):
    axes[1].text(bar.get_x() + bar.get_width()/2, v, 
                f'${v:,.0f}', ha='center', 
                va='bottom' if v > 0 else 'top', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Save Best Model and Generate Report

In [ ]:
# Save best model
trainer.save_model(best_model, 'best_model.pkl')
print(f"Best model ({best_model_name}) saved successfully!")

In [ ]:
# Generate comprehensive report
generate_model_report(results_df, business_metrics, output_file='model_report.txt')
print("\nComprehensive model report generated!")

## 10. Model Deployment Recommendations

In [ ]:
print("\n" + "="*80)
print("MODEL DEPLOYMENT RECOMMENDATIONS")
print("="*80)

print(f"\n1. BEST MODEL: {best_model_name}")
print(f"   - Accuracy: {results_df.iloc[0]['accuracy']:.4f}")
print(f"   - F1-Score: {results_df.iloc[0]['f1_score']:.4f}")
print(f"   - ROC-AUC: {results_df.iloc[0].get('roc_auc', 'N/A')}")

print("\n2. KEY INSIGHTS:")
print("   - Contract type is a strong predictor of churn")
print("   - Customers with shorter tenure are at higher risk")
print("   - Fiber optic customers show higher churn rates")
print("   - Payment method affects churn probability")

print("\n3. BUSINESS RECOMMENDATIONS:")
print("   - Prioritize retention for month-to-month contract customers")
print("   - Offer incentives for long-term contracts")
print("   - Focus on customers in first 12 months")
print("   - Improve fiber optic service quality")
print("   - Encourage automatic payment methods")

print("\n4. DEPLOYMENT STRATEGY:")
print("   - Deploy model via Streamlit dashboard")
print("   - Set up automated weekly predictions")
print("   - Monitor model performance monthly")
print("   - Retrain model quarterly with new data")
print("   - A/B test retention strategies")

print("\n" + "="*80)
print("MODELING COMPLETE! Ready for dashboard deployment.")
print("="*80)